# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [ ]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [ ]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [ ]:
# The .env file lives in this folder (project/starter) and must contain:
# OPENAI_API_KEY="voc-..."
# OPENAI_BASE_URL="https://openai.vocareum.com/v1"
# TAVILY_API_KEY="tvly-..."
# See .env.example for the variable names.

In [ ]:
from dotenv import find_dotenv

# find_dotenv() finds the .env file in this folder (or a parent folder);
# override=True makes the values in .env win over anything already set.
load_dotenv(find_dotenv(), override=True)

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY is not set"
print("Environment variables loaded")

### VectorDB Instance

In [ ]:
# Part 2 loads exactly this path, so keep it as "chromadb"
chroma_client = chromadb.PersistentClient(path="chromadb")

### Collection

In [ ]:
# OpenAI embeddings, reading the key from OPENAI_API_KEY (not CHROMA_OPENAI_API_KEY).
# Part 2 must rebuild this same function when it loads the collection.
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key_env_var="OPENAI_API_KEY",
    api_base=os.getenv("OPENAI_BASE_URL"),
)

In [ ]:
# get_or_create_collection lets this notebook be re-run without errors
collection = chroma_client.get_or_create_collection(
    name="udaplay",
    embedding_function=embedding_fn,
)

### Add documents

In [ ]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    # Genre and Publisher are indexed too, so questions like
    # "Who published ...?" can be matched semantically
    content = (
        f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - "
        f"Genre: {game['Genre']}. Publisher: {game['Publisher']}. "
        f"{game['Description']}"
    )

    # Use file name (like 001) as ID
    doc_id = os.path.splitext(file_name)[0]

    # upsert instead of add: re-running the notebook does not create duplicates
    collection.upsert(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

print(f"Documents in collection '{collection.name}': {collection.count()}")

### Semantic search

Query the collection with natural-language questions. A lower distance means a closer match.

In [ ]:
def search_games(query: str, n_results: int = 3):
    """Semantic search over the udaplay collection."""
    results = collection.query(
        query_texts=[query],
        n_results=n_results,
        include=["documents", "metadatas", "distances"],
    )
    return [
        {"id": doc_id, "distance": round(distance, 4), **metadata}
        for doc_id, metadata, distance in zip(
            results["ids"][0], results["metadatas"][0], results["distances"][0]
        )
    ]


def print_results(query: str, n_results: int = 3):
    print(f"Query: {query}")
    for r in search_games(query, n_results):
        print(
            f"  [{r['id']}] {r['Name']} | {r['Platform']} | {r['YearOfRelease']} | "
            f"{r['Publisher']} | distance={r['distance']}"
        )
    print()

In [ ]:
queries = [
    "When was Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Who published Grand Theft Auto: San Andreas?",
    "Racing games on PlayStation",
    "Was Mortal Kombat X released for PlayStation 5?",  # not in the dataset
]

for q in queries:
    print_results(q)

### Reload check

Part 2 opens the database from disk with the same path, collection name and embedding function. This cell does the same with a fresh client to confirm the data persisted.

In [ ]:
reloaded_client = chromadb.PersistentClient(path="chromadb")
reloaded = reloaded_client.get_collection("udaplay", embedding_function=embedding_fn)
print(f"Reloaded collection '{reloaded.name}' with {reloaded.count()} documents")
print(reloaded.peek(1)["documents"][0])